# Error decomposition: coarse vs reference
Code: `src/constrain_cleaning/pencil_error.py` (tests: `tests/test_pencil_error.py`).

### Definitions
| symbol | definition | code |
|---|---|---|
| $c_{\mathbf k}[f]$ | $f(\mathbf x)=\sum_{\mathbf k}c_{\mathbf k}e^{i\mathbf k\cdot\mathbf x}$, phase-corrected for the cell-centred grid $x_j=-\pi+(j+\frac12)\delta x$ | `coeffs(f, x0)` |
| $P$ | keep $\vert k_i\vert<N/2$ of the reference ($N=64$, reference $128^3$) | `restrict(c, N)` |
| $e$ | $x_N-Px_R$ | `decompose(...)["e2"]` = $\langle e^2\rangle$ per field |
| $v_G$ | $C_G(x_N)=i\mathbf k\cdot\hat{\mathbf E}-\hat J^0$ | `vG` |
| $v_A$ | $C_A(x_N)=\hat\Gamma-i\mathbf k\cdot\hat{\mathbf A}$ | `vA` |
| $\Vert\cdot\Vert^2$ | $\langle f^2\rangle=\sum_{\mathbf k}\vert c_{\mathbf k}\vert^2$ | `norm2` |

### Cleaning from the violation alone
$$\min\ \Vert\delta\mathbf E\Vert^2+\lambda^2\Vert\delta J^0\Vert^2\quad\text{s.t.}\quad C_G(x_N+\delta x)=0:\qquad
\delta\hat{\mathbf E}=-\frac{v_G\,(i\mathbf k)^*}{k^2+\lambda^{-2}},\qquad \delta\hat J^0=\frac{v_G\,\lambda^{-2}}{k^2+\lambda^{-2}}$$
$$\min\ \Vert\delta\Gamma\Vert^2+\mu^2\Vert\delta\mathbf A\Vert^2\quad\text{s.t.}\quad C_A=0:\qquad
\delta\hat\Gamma=-\frac{v_A}{1+k^2/\mu^2},\qquad \delta\hat{\mathbf A}=-\frac{v_A\,i\mathbf k}{\mu^2(1+k^2/\mu^2)}$$

| variant | weights | changes |
|---|---|---|
| E-only | $\lambda=\infty$ | $\mathbf E_\parallel$ so that $\nabla\cdot\mathbf E=J^0$ |
| J0-only | $\lambda=0$ | $J^0\leftarrow\nabla\cdot\mathbf E$ |
| mixed | $\lambda=1$ | both |
| A-only | $\mu=0$ | $\mathbf A_\parallel$ so that $\nabla\cdot\mathbf A=\Gamma$ |
| Gamma-only | $\mu=\infty$ | $\Gamma\leftarrow\nabla\cdot\mathbf A$ |

Finite weights: orthogonal projection, and with $C(Px_R)\approx0$
$$\Vert e_{\rm after}\Vert^2=\Vert e\Vert^2-\Vert e_\perp\Vert^2,\qquad \Vert e_\perp\Vert^2=\sum_{\mathbf k}\frac{\vert v_G\vert^2}{k^2+\lambda^{-2}}$$
One-sided variants: not orthogonal projections of $e$; evaluated explicitly ($e_{\rm after}=e+\delta x$).

### Ceiling
$$f_{\rm state}=\frac{\Vert e_\perp\Vert^2_{\lambda=1}}{\sum_{\rm fields}\Vert e\Vert^2}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, LogLocator, NullLocator

ROOT = Path.home() / "projects/Constrain_cleaning"
sys.path.insert(0, str(ROOT / "src"))
from constrain_cleaning import pencil_error as pe
X = ROOT / "runs/pencil/experiments"

mpl.rcParams.update({
    "figure.figsize": (9, 6), "figure.dpi": 110, "font.size": 16, "axes.titlesize": 17, "axes.labelsize": 17,
    "xtick.labelsize": 15, "ytick.labelsize": 15, "legend.fontsize": 13, "lines.linewidth": 2.8,
    "lines.markersize": 9, "axes.linewidth": 1.5, "axes.grid": True, "grid.alpha": 0.3,
    "xtick.major.size": 7, "ytick.major.size": 7, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
})
def style(fig):
    for ax in fig.axes:
        for axis, scale in [(ax.xaxis, ax.get_xscale()), (ax.yaxis, ax.get_yscale())]:
            if scale == "log":
                axis.set_major_locator(LogLocator(base=10, numticks=5)); axis.set_minor_locator(NullLocator())
            else:
                axis.set_major_locator(MaxNLocator(nbins=5))
    fig.tight_layout()

# coarse 64^3 run -> (reference 128^3, 32^3 for the reference-quality check)
PAIRS = {
    "A/rel":           ("R/A_rel_N128", "R/A_rel_N032"),
    "A/galilean":      ("R/A_galilean_N128", "R/A_galilean_N032"),
    "G/w2":            ("R/G_w2_N128", "R/G_w2_N032"),
    "G/w0_divE":       ("R/G_w0_divE_N128", "R/G_w0_divE_N032"),
    "E/sig1_N064":     ("E/sig1_N128", "E/sig1_N032"),
    "E/sig1_noenf_N064": ("E/sig1_noenf_N128", "E/sig1_noenf_N032"),
    "E/underres_N064": ("E/underres_N128", "E/underres_N032"),
}
done = lambda r: (X / r / "COMPLETED").exists()
PAIRS = {c: rr for c, rr in PAIRS.items() if done(c) and done(rr[0])}
print("pairs:", list(PAIRS))
GROUPS = {"u": pe.VEC["u"], "A": pe.VEC["A"], "E": pe.VEC["E"], "J0": ("rhoe",), "lnrho": ("lnrho",), "Gamma": ("Gamma",)}

### Decomposition at every common snapshot
`R[run]` = list over snapshots of `decompose(x_64, x_128)`; `R32[run]` = `decompose(x_32, x_64)`.

In [ ]:
def nsnap(r):
    return len(list((X / r / "data" / "proc0").glob("VAR[0-9]*")))

R, R32 = {}, {}
for c, (ref, c32) in PAIRS.items():
    n = min(nsnap(c), nsnap(ref))
    R[c], R32[c] = [], []
    for k in range(1, n):          # VAR0 lacks the divE aux and cannot be read
        x64, x128 = pe.read_state(X / c, k), pe.read_state(X / ref, k)
        assert abs(x64["t"] - x128["t"]) < 1e-9, (c, k, x64["t"], x128["t"])
        R[c].append(pe.decompose(x64, x128))
        if done(c32) and k < nsnap(c32):
            R32[c].append(pe.decompose(pe.read_state(X / c32, k), x64))
    print(f"{c:20s} {len(R[c])} snapshots, t = {[round(o['t'], 3) for o in R[c]]}")
T = {c: np.array([o["t"] for o in R[c]]) for c in R}
gsum = lambda o, key, fs: sum(o[key][f] for f in fs)

## Checks
| check | condition |
|---|---|
| reference quality | $\Vert x_{32}-Px_{64}\Vert/\Vert x_{64}-Px_{128}\Vert\approx2^{p}$, $p=5$–$6$ |
| reference violation | $\Vert C_G(Px_R)\Vert^2/\Vert v_G\Vert^2\ll1$ |
| Pythagoras ($\lambda=\mu=1$) | $\Vert e_{\rm after}\Vert^2/(\Vert e\Vert^2-\Vert e_\perp\Vert^2)\approx1$ |

In [ ]:
for c in R:
    print(f"{c}")
    for i, o in enumerate(R[c]):
        line = f"  t={o['t']:5.3f}  vG_ref/vG={o['vG2_ref'] / o['vG2']:.1e}"
        g = o["gauss"][1.0]
        line += f"  pythagoras={g['metric_after'] / (g['metric_before'] - g['eperp2']):.4f}"
        if i < len(R32[c]):
            o32 = R32[c][i]
            line += "  |e32|/|e64|: " + " ".join(
                f"{gname}={np.sqrt(gsum(o32, 'e2', fs) / gsum(o, 'e2', fs)):4.0f}" for gname, fs in GROUPS.items())
        print(line)
    assert all(o["vG2_ref"] / o["vG2"] < 1e-2 for o in R[c]), f"{c}: reference violates the constraint"
print("reference violation << coarse violation for all pairs -> OK")

## Relative error per field
$\Vert e\Vert/\Vert Px_R\Vert$ vs $t$.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(24, 13), sharex=True)
for ax, (gname, fs) in zip(axes.flat, GROUPS.items()):
    for j, c in enumerate(R):
        y = [np.sqrt(gsum(o, "e2", fs) / max(gsum(o, "x2", fs), 1e-300)) for o in R[c]]
        ax.semilogy(T[c], y, "o-", color=plt.cm.tab10(j), label=c)
    ax.set_title(f"{gname}: |e|/|x|"); ax.set_xlabel("t")
axes.flat[0].legend(fontsize=12)
style(fig)

## Cleaning: $\Vert e_{\rm after}\Vert^2/\Vert e\Vert^2$
Below 1: cleaning moves the coarse state towards the reference. Above 1: away from it.

In [ ]:
VARIANTS = [("E-only (λ=∞)", "gauss", np.inf, "eE2", pe.VEC["E"]),
            ("J0-only (λ=0)", "gauss", 0.0, "eJ2", ("rhoe",)),
            ("mixed (λ=1): |E|²+|J0|²", "gauss", 1.0, "metric", None),
            ("A-only (μ=0)", "gamma", 0.0, "eA2", pe.VEC["A"]),
            ("Γ-only (μ=∞)", "gamma", np.inf, "eG2", ("Gamma",)),
            ("mixed (μ=1): |Γ|²+|A|²", "gamma", 1.0, "metric", None)]
fig, axes = plt.subplots(2, 3, figsize=(24, 13), sharex=True)
for ax, (title, grp, w, key, fs) in zip(axes.flat, VARIANTS):
    for j, c in enumerate(R):
        if key == "metric":
            y = [o[grp][w]["metric_after"] / o[grp][w]["metric_before"] for o in R[c]]
        else:
            y = [o[grp][w][key] / gsum(o, "e2", fs) for o in R[c]]
        ax.semilogy(T[c], y, "o-", color=plt.cm.tab10(j), label=c)
    ax.axhline(1, color="k", lw=1.5, ls="--")
    ax.set_title(title); ax.set_xlabel("t")
axes.flat[0].legend(fontsize=12)
style(fig)

## Ceiling
$f_{\rm state}=\Vert e_\perp\Vert^2_{\lambda=1}/\sum_{\rm fields}\Vert e\Vert^2$; physical fields $(\mathbf u,\mathbf A,\mathbf E,\ln\rho)$ are unchanged by J0-only and Γ-only cleaning.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
print(f"{'run':20s} {'t':>6s} {'f_state':>10s} {'J0 share of |e|^2':>18s} {'Gamma share':>12s}")
for j, c in enumerate(R):
    f = [o["gauss"][1.0]["eperp2"] / sum(o["e2"].values()) for o in R[c]]
    ax.semilogy(T[c], f, "o-", color=plt.cm.tab10(j), label=c)
    for o in R[c][::2]:
        tot = sum(o["e2"].values())
        print(f"{c:20s} {o['t']:6.3f} {o['gauss'][1.0]['eperp2'] / tot:10.2e} {o['e2']['rhoe'] / tot:18.2e} {o['e2']['Gamma'] / tot:12.2e}")
ax.set_xlabel("t"); ax.set_ylabel(r"$f_{\rm state}$"); ax.legend(fontsize=12)
style(fig)

## Spectra (last common snapshot)
$\Vert e\Vert^2(k)$ per field and $\Vert e_\perp\Vert^2(k)$, shell sums.

In [ ]:
runs = list(R)
fig, axes = plt.subplots(2, (len(runs) + 1) // 2, figsize=(8.5 * ((len(runs) + 1) // 2), 13), squeeze=False)
for ax, c in zip(axes.flat, runs):
    o = R[c][-1]
    for j, (name, s) in enumerate(o["spec_e"].items()):
        kk = np.arange(s.size); m = (kk >= 1) & (s > 0)
        ax.loglog(kk[m], s[m], color=plt.cm.tab10(j), label=f"|e_{name}|²")
    s = o["spec_vG"] / np.maximum(np.arange(o["spec_vG"].size) ** 2, 1)   # shell sum of |v_G|^2, / k^2
    kk = np.arange(s.size); m = (kk >= 1) & (s > 0)
    ax.loglog(kk[m], s[m], "k--", lw=2, label="|v_G|²/k²")
    ax.set_title(f"{c}, t={o['t']:.2f}"); ax.set_xlabel("k")
for ax in axes.flat[len(runs):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=11)
style(fig)

## Summary
Time averages over snapshots.

In [ ]:
print(f"{'run':20s}{'|e_u|/|u|':>11s}{'|e_E|/|E|':>11s}{'f_state':>10s}{'E-only':>9s}{'J0-only':>9s}{'mixed':>8s}{'A-only':>9s}{'Γ-only':>9s}")
for c in R:
    m = lambda fn: np.mean([fn(o) for o in R[c]])
    print(f"{c:20s}"
          f"{m(lambda o: np.sqrt(gsum(o, 'e2', pe.VEC['u']) / gsum(o, 'x2', pe.VEC['u']))):11.2e}"
          f"{m(lambda o: np.sqrt(gsum(o, 'e2', pe.VEC['E']) / gsum(o, 'x2', pe.VEC['E']))):11.2e}"
          f"{m(lambda o: o['gauss'][1.0]['eperp2'] / sum(o['e2'].values())):10.2e}"
          f"{m(lambda o: o['gauss'][np.inf]['eE2'] / gsum(o, 'e2', pe.VEC['E'])):9.3f}"
          f"{m(lambda o: o['gauss'][0.0]['eJ2'] / o['e2']['rhoe']):9.3f}"
          f"{m(lambda o: o['gauss'][1.0]['metric_after'] / o['gauss'][1.0]['metric_before']):8.3f}"
          f"{m(lambda o: o['gamma'][0.0]['eA2'] / gsum(o, 'e2', pe.VEC['A'])):9.3f}"
          f"{m(lambda o: o['gamma'][np.inf]['eG2'] / o['e2']['Gamma']):9.3f}")

## Results (64³ vs 128³; time averages over snapshots)
| run | $\sigma$ | enforcement | $\Vert e_u\Vert/\Vert u\Vert$ | $\Vert e_E\Vert/\Vert E\Vert$ | $f_{\rm state}$ | E-only | J0-only | mixed $\lambda=1$ | A-only | Γ-only |
|---|---|---|---|---|---|---|---|---|---|---|
| `A/rel` | 11.9 | default | $1.1\times10^{-7}$ | $8.6\times10^{-8}$ | $7\times10^{-4}$ | 1.32 | **0.32** | 0.93 | 1.88 | **0.10** |
| `G/w2` | 11.9 | default | $7.7\times10^{-8}$ | $6.8\times10^{-8}$ | $8\times10^{-5}$ | 1.03 | 0.98 | 0.99 | 8.27 | **0.10** |
| `G/w0_divE` | 11.9 | none | $7.7\times10^{-8}$ | $1.8\times10^{-7}$ | 0.10 | 250 | **0.007** | 0.89 | 4.46 | **0.26** |
| `E/sig1` | 1 | default | $4.6\times10^{-8}$ | $4.2\times10^{-8}$ | $1.2\times10^{-3}$ | **0.93** | 1.36 | 0.96 | 5.80 | **0.04** |
| `E/sig1_noenf` | 1 | none | $4.5\times10^{-8}$ | $6.3\times10^{-7}$ | 0.17 | **0.67** | 2.92 | **0.55** | 2.38 | **0.45** |
| `E/underres` | 11.9 | default, $\nu=5\times10^{-3}$ | $1.1\times10^{-6}$ | $3.7\times10^{-7}$ | $1.0\times10^{-3}$ | 1.24 | **0.42** | 0.93 | 1.77 | **0.19** |

Cleaning columns: $\Vert e_{\rm after}\Vert^2/\Vert e\Vert^2$ (bold: $<1$).

| regime | carrier of the Gauss violation | best correction |
|---|---|---|
| $\sigma=11.9$ | $J^0$ ($\mathbf E$ slaved to Ohm's law, relaxation rate $\gamma\sigma$) | $J^0\leftarrow\nabla\cdot\mathbf E$ |
| $\sigma=1$ | $\mathbf E_\parallel$ ($\mathbf E$ dynamical) | $\mathbf E_\parallel$ from $J^0$, or mixed |
| all | $\Gamma$ (not $\mathbf A$) | $\Gamma\leftarrow\nabla\cdot\mathbf A$ |

Static cleaning never changes $\mathbf u$; $\Vert e_u\Vert$ is the same with and without enforcement ($\sigma=1$: $4.6$ vs $4.5\times10^{-8}$), while $\Vert e_E\Vert$ is $15\times$ larger without enforcement.
